In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv('/workspaces/Deep-learning-for-condominium-price-prediction-using-heterogeneous-analysis/input_model/tabular_data/final_input_condo_poi_data.csv')

print(f"ขนาดข้อมูล: {df.shape}")
df.head()

2025-12-10 01:12:32.956163: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-10 01:12:57.311261: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-10 01:13:09.465903: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


ขนาดข้อมูล: (14722, 75)


,id,name,district,lat,long,sale_price,asking_price,asking_price_change_quater,asking_price_change_year,gross_rental_yield,...,gov_service_1000m,Hospital_1000m,mall_1000m,night_club_1000m,public_park_1000m,restaurant_1000m,supermarket_1000m,Airport_1000m,E_railway_1000m,railway_1000m
0,1041,Suan Thon Park Condo,Thung Khru,13.652504,100.487825,1650000,23042,0.0,0.0,5.4,...,0,0,0,1,0,6,1,0,0,0
1,795,Origins Rama 2,Chom Thong,13.654157,100.423897,1550000,48384,0.0,0.0,4.5,...,0,0,0,1,0,1,0,0,0,0
2,795,Origins Rama 2,Chom Thong,13.654157,100.423897,1187000,48384,0.0,0.0,4.5,...,0,0,0,1,0,1,0,0,0,0
3,795,Origins Rama 2,Chom Thong,13.654157,100.423897,2600000,48384,0.0,0.0,4.5,...,0,0,0,1,0,1,0,0,0,0
4,795,Origins Rama 2,Chom Thong,13.654157,100.423897,1187000,48384,0.0,0.0,4.5,...,0,0,0,1,0,1,0,0,0,0


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings

# ปิด Warning เพื่อความสะอาดของหน้าจอ
warnings.filterwarnings('ignore')

# --- 1. Import Models ---
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
import xgboost as xgb
import lightgbm as lgb

# Tools
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

# ==========================================
# 2. เตรียมข้อมูล (Data Preparation)
# ==========================================
print("กำลังเตรียมข้อมูล...")

# เลือก Features และ Target
drop_cols = ['sale_price', 'name', 'id']
existing_drop = [c for c in drop_cols if c in df.columns]

X = df.drop(columns=existing_drop)
y = np.log1p(df['sale_price']) # Log Transform

# แบ่ง Train/Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessor
categorical_cols = ['district']
numerical_cols = [c for c in X.columns if c not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

# ==========================================
# 3. ตั้งค่า Grid Search (Super Fine-Tuning)
# ==========================================
# เพิ่ม Parameter ให้เป็น 4 ระดับ (หรือมากกว่า) ในแต่ละตัว
model_params = {
    # --- กลุ่ม Linear ---
    'Ridge': {
        'model': Ridge(),
        'params': {
            'regressor__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]
        }
    },
    'ElasticNet': {
        'model': ElasticNet(),
        'params': {
            'regressor__alpha': [0.01, 0.1, 1.0, 10.0],
            'regressor__l1_ratio': [0.2, 0.4, 0.6, 0.8]
        }
    },

    # --- กลุ่ม Distance ---
    'KNN': {
        'model': KNeighborsRegressor(),
        'params': {
            'regressor__n_neighbors': [3, 5, 10, 20, 30],
            'regressor__weights': ['uniform', 'distance'],
            'regressor__p': [1, 2] # 1=Manhattan, 2=Euclidean
        }
    },

    # --- กลุ่ม Tree Ensemble ---
    'RandomForest': {
        'model': RandomForestRegressor(random_state=42),
        'params': {
            'regressor__n_estimators': [100, 200, 300, 500],
            'regressor__max_depth': [10, 20, 30, None],
            'regressor__min_samples_leaf': [1, 2, 4, 8],
            'regressor__max_features': ['sqrt', 'log2', 0.5]
        }
    },
    'ExtraTrees': {
        'model': ExtraTreesRegressor(random_state=42),
        'params': {
            'regressor__n_estimators': [100, 200, 300, 500],
            'regressor__max_depth': [10, 20, 30, None],
            'regressor__min_samples_leaf': [1, 2, 4, 8]
        }
    },

    # --- กลุ่ม Boosting (ตัวเก็งชนะเลิศ) ---
    'GradientBoosting': {
        'model': GradientBoostingRegressor(random_state=42),
        'params': {
            'regressor__n_estimators': [100, 200, 300, 500],
            'regressor__learning_rate': [0.01, 0.05, 0.1, 0.2],
            'regressor__max_depth': [3, 4, 5, 6]
        }
    },
    'XGBoost': {
        'model': xgb.XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1),
        'params': {
            'regressor__n_estimators': [500, 1000, 2000, 3000],
            'regressor__learning_rate': [0.005, 0.01, 0.05, 0.1],
            'regressor__max_depth': [3, 5, 7, 9],
            'regressor__subsample': [0.6, 0.7, 0.8, 0.9]
        }
    },
    'LightGBM': {
        'model': lgb.LGBMRegressor(random_state=42, verbose=-1),
        'params': {
            'regressor__n_estimators': [500, 1000, 2000, 3000],
            'regressor__learning_rate': [0.005, 0.01, 0.05, 0.1],
            'regressor__num_leaves': [31, 50, 100, 200], # ยิ่งเยอะยิ่งฉลาด
            'regressor__feature_fraction': [0.6, 0.7, 0.8, 0.9]
        }
    }
}

# ==========================================
# 4. เริ่มรัน (The Ultimate Battle)
# ==========================================
results = []
print(f"เริ่มการประลอง 8 อัลกอริทึม (แบบ Super Fine-Tuning)...")
print("หมายเหตุ: เนื่องจากพารามิเตอร์เยอะมาก อาจใช้เวลานาน โปรดรอสักครู่ครับ...")
print("="*70)

for model_name, mp in model_params.items():
    start_time = time.time()
    
    clf = Pipeline(steps=[('preprocessor', preprocessor),
                          ('regressor', mp['model'])])
    
    # Grid Search (cv=3)
    grid = GridSearchCV(clf, mp['params'], cv=3, scoring='r2', n_jobs=-1)
    
    print(f"Training {model_name}...")
    try:
        grid.fit(X_train, y_train)
        
        best_model = grid.best_estimator_
        y_pred_log = best_model.predict(X_test)
        y_pred = np.expm1(y_pred_log)
        y_true = np.expm1(y_test)
        
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
        
        duration = time.time() - start_time
        
        results.append({
            'Model': model_name,
            'Best R2': r2,
            'MAE (Baht)': mae,
            'MAPE (%)': mape,
            'Best Params': grid.best_params_,
            'Training Time (s)': duration
        })
        print(f"  --> Done! R2: {r2:.4f} | MAE: {mae:,.0f} | MAPE: {mape:.2f}% | Used: {duration:.1f}s")
        
    except Exception as e:
        print(f"  --> Error with {model_name}: {e}")

# ==========================================
# 5. สรุปผล
# ==========================================
results_df = pd.DataFrame(results).sort_values(by='MAE (Baht)', ascending=True)

print("\n" + "="*70)
print("🏆 Leaderboard: สรุปผลการเปรียบเทียบ (เรียงตามความแม่นยำ)")
print("="*70)
print(results_df[['Model', 'MAE (Baht)', 'MAPE (%)', 'Best R2', 'Training Time (s)']].to_string(index=False))

# Visualization
plt.figure(figsize=(12, 6))
sns.barplot(x='MAE (Baht)', y='Model', data=results_df, palette='viridis')
plt.title('Model Comparison: Mean Absolute Error (Lower is Better)')
plt.xlabel('MAE (Baht)')
plt.show()

if not results_df.empty:
    winner = results_df.iloc[0]
    print(f"\n🥇 ผู้ชนะเลิศคือ: {winner['Model']}")
    print("สุดยอดพารามิเตอร์ (Best Params):")
    print(winner['Best Params'])

กำลังเตรียมข้อมูล...
เริ่มการประลอง 8 อัลกอริทึม (แบบ Super Fine-Tuning)...
หมายเหตุ: เนื่องจากพารามิเตอร์เยอะมาก อาจใช้เวลานาน โปรดรอสักครู่ครับ...
Training Ridge...
  --> Done! R2: 0.4443 | MAE: 1,181,666 | MAPE: 18.22% | Used: 4.8s
Training ElasticNet...
  --> Done! R2: 0.3917 | MAE: 1,242,474 | MAPE: 19.36% | Used: 1.8s
Training KNN...
  --> Done! R2: 0.9033 | MAE: 711,745 | MAPE: 11.67% | Used: 74.2s
Training RandomForest...
  --> Done! R2: 0.9450 | MAE: 606,026 | MAPE: 10.23% | Used: 1873.1s
Training ExtraTrees...
  --> Done! R2: 0.9432 | MAE: 608,570 | MAPE: 10.32% | Used: 1735.8s
Training GradientBoosting...
  --> Done! R2: 0.9452 | MAE: 613,373 | MAPE: 10.44% | Used: 1377.1s
Training XGBoost...
  --> Done! R2: 0.9471 | MAE: 608,568 | MAPE: 10.25% | Used: 4365.0s
Training LightGBM...
